# Module 05 — Lists and tuples

Module 01 wrote `b = a` with a number and left a promise: the answer changes when
`a` is a list. Everything below is that one fact and its consequences.

## 1. A list is not an array

No fixed size, no element type, no `new`. `[]` is the literal, `append` grows it,
and negative indices count from the end.

In [ ]:
log = []
log.append(21.7)
log.append(23.1)
log.append("TH-04 offline")  # nothing stops you -- there is no element type

print(log)
print(len(log), log[0], log[-1])

Mixing types is legal and usually a design error: the code that reads `log` now has
to cope with both. What it buys you is `["TH-04", 21.7]` — a record of two fields,
which is a job for a tuple (section 7).

Reading past the end raises `IndexError` rather than handing you whatever was in
memory. That is the first of several places in this module where Python is louder
than C.

## 2. Two names, one list

`b = a` copies the reference, not the object — exactly as it did in module 01. The
difference is that a list can change, so the sharing becomes visible.

Predict both, then run the cell.

In [ ]:
a = [1, 2]
b = a
b.append(3)

x = 1
y = x
y += 1

assert a == ...
assert x == ...

Nothing special happened to `b = a`. `y = x` shared the `1` in exactly the same way
— but an `int` has no method that changes it, so the sharing never showed. `y += 1`
made a new object and rebound `y`.

`is` answers the question directly: it compares identity, not value.

In [ ]:
a = [1, 2]
b = a
c = [1, 2]

print(a is b, a is c, a == c)
print(id(a) == id(b))  # id() is the identity `is` compares -- an address, in CPython

In Java, `List<Integer> b = a` behaves the same way and you know it. What is new is
that Python applies the rule to *everything*, with no `int`/`Integer` split to warn
you which side of the line you are on.

## 3. `+=` mutates a list and rebinds a tuple

The same operator, two behaviours. This is the sharpest version of section 2, and
the one that bites in real code.

In [ ]:
lst = [1, 2]
lst_alias = lst
lst += [3]

t = (1, 2)
t_alias = t
t += (3,)

# One of the two aliases has seen the change. Which one?
assert lst_alias == ...
assert t_alias == ...

`lst += [3]` calls the list's in-place add, which appends to the existing object and
returns it. The name is rebound to the object it already had, so the alias sees the
new element.

A tuple has no in-place add. `t += (3,)` falls back on `t = t + (3,)`, which builds
a **new** tuple and rebinds `t` alone. `t_alias` still refers to the old one.

So `+=` on a list is `extend`, and `+=` on a tuple is `=`. Reading the line tells
you neither; you have to know what is on the left.

In [ ]:
lst = [1, 2]
before = lst
lst += [3]
print("list  :", lst, "| same object as before:", lst is before)

t = (1, 2)
before = t
t += (3,)
print("tuple :", t, "| same object as before:", t is before)

## 4. Slicing

`seq[start:stop:step]` — `start` included, `stop` excluded, all three optional. C
and Java have no counterpart at all; the nearest thing is a loop or
`Arrays.copyOfRange`.

In [ ]:
v = [0, 1, 2, 3, 4, 5]

print(v[1:4])  # from 1 up to but not including 4
print(v[:2])  # from the start
print(v[-3:])  # the last three
print(v[::2])  # every second
print(v[::-1])  # reversed
print(v[10:])  # out of range gives [], not an error -- unlike v[10]

`stop` being exclusive is what makes `v[:k]` and `v[k:]` fit together with no
overlap and no gap: every entry is in exactly one of the two, for any `k`. Where the
slice lies inside the list, its length is `stop - start` — but only there, because a
slice that runs past the end stops at the end instead of complaining.

A slice always builds a **new** list. `v[:]` is therefore a copy — and a shallow
one, which is section 5.

In [ ]:
v = [0, 1, 2, 3, 4, 5]
copy_of_v = v[:]
print(copy_of_v == v, copy_of_v is v)

# A slice also works on the left of `=`, and the piece you put in need not be
# the same length as the piece it replaces.
w = [1, 2, 3, 4]
w[1:3] = [9, 9, 9]
print(w)

del w[0:2]
print(w)

## 5. Copying is shallow

`v[:]`, `list(v)`, `v.copy()` and `copy.copy(v)` all do the same thing: a new outer
list, holding **the same objects** as the old one. If those objects are themselves
lists, nothing about them was copied.

In [ ]:
import copy

readings = [[21.7], [23.1]]
shallow = copy.copy(readings)

shallow[0].append(99.9)  # touching the copy...

print("copy    :", shallow)
print("original:", readings)  # ...and the original changed too
print("shared row:", shallow[0] is readings[0])

`copy.deepcopy` walks the structure and copies every level.

In [ ]:
import copy

readings = [[21.7], [23.1]]
deep = copy.deepcopy(readings)

deep[0].append(99.9)

print("copy    :", deep)
print("original:", readings)
print("shared row:", deep[0] is readings[0])

`deepcopy` is not free — it walks everything and keeps track of what it has already
seen, so that a structure containing itself does not loop forever. Reach for it when
you actually have nesting; for a flat list of numbers or strings, `v[:]` is the
whole answer.

## 6. `[[0] * 3] * 3`

`*` on a list repeats it — and repeats the *reference*, which is section 5 in its
most expensive form. This is a grid that does not work.

In [ ]:
grid = [[0] * 3] * 3
grid[0][0] = 9

# Three rows. One assignment. What does grid look like?
assert grid == ...

`[0] * 3` built one row. `[row] * 3` then put that one row into the outer list three
times. There is one row object and three references to it.

The fix is a comprehension, because it evaluates `[0] * 3` again on every pass:

In [ ]:
grid = [[0] * 3 for _ in range(3)]
grid[0][0] = 9

print(grid)
print("rows are distinct:", grid[0] is not grid[1])

`_` is an ordinary name that says "I am not going to use this". Nothing enforces
that; it is a convention, and a strong one.

## 7. Tuples

A tuple is not a read-only list. It is a value with a fixed shape, where position
carries meaning: `("TH-04", 21.7)` is *a reading*, and its two slots are not
interchangeable. A list holds an unknown number of interchangeable things.

The comma makes the tuple, not the parentheses — `t = 1, 2` is a tuple. The
parentheses are for grouping, which is why a one-element tuple needs the trailing
comma: `(1,)`.

In [ ]:
reading = "TH-04", 21.7
print(reading, type(reading))

try:
    reading[0] = "TH-09"
except TypeError as err:
    print("TypeError:", err)

# But immutable means the tuple's slots cannot be rebound -- not that the objects
# in them cannot change.
mixed = ([1], 2)
mixed[0].append(9)
print(mixed)

Unpacking works on any sequence and is the everyday reason tuples turn up: a
function returns several values, and the call site takes them apart.

In [ ]:
tag, value = ("TH-04", 21.7)
print(tag, value)

first, *rest = [1, 2, 3, 4]  # * collects what is left over, always into a list
print(first, rest)

start, *middle, end = [1, 2, 3, 4]
print(start, middle, end)

a, b = 1, 2
a, b = b, a  # no temporary: the right-hand side is built first
print(a, b)

The count has to match, or you get a `ValueError` — `too many values to unpack
(expected 2)`. That is a real check, unlike C, where nothing corresponds to it.

## 8. `x.sort()` returns `None`

Every list method that changes the list returns `None`: `sort`, `reverse`, `append`,
`extend`, `insert`. `pop` is the exception, and it returns the item it removed, not
the list.

In [ ]:
readings = [23.1, 21.7, 91.0]
result = readings.sort()

assert result is ...
assert readings == ...

This is deliberate: a method either changes the object or answers a question about
it, never both. So `x.sort()` cannot be chained, and `y = x.sort()` gives you
`None` instead of a quietly shared list.

When you want a sorted result and the original left alone, `sorted` builds a new
list. It accepts any iterable — a tuple, a `dict`, the generators of module 13 — and
always hands back a list.

In [ ]:
readings = [23.1, 21.7, 91.0]

print(sorted(readings))
print(readings)  # untouched

log = [("TH-04", 91.0), ("TH-01", 21.7)]
print(sorted(log))  # tuples compare element by element
print(sorted(log, key=lambda item: item[1]))
print(sorted(log, key=lambda item: item[1], reverse=True))

## 9. Comprehensions

`[expression for name in sequence if condition]` — the idiom that replaces most of
the loops you would write in C. It builds a new list; it does not modify anything.

In [ ]:
readings = [21.7, 23.1, 91.0, 19.4]

print([round(r) for r in readings])
print([r for r in readings if r > 22])
print([(i, r) for i, r in enumerate(readings) if r > 22])

One detail with no equivalent in a `for` loop: the loop variable of a comprehension
lives only inside it.

In [ ]:
for i in range(3):
    pass

print("after the loop, i is", i)  # the loop variable outlives the loop

squares = [n * n for n in range(3)]

try:
    print(n)
except NameError as err:
    print("NameError:", err)  # the comprehension had its own scope

A comprehension is the right choice when it says *one* thing: map, filter, or both.
When the body needs several statements, a condition with two branches, or an early
exit, a loop reads better and you should write one. A comprehension that no longer
fits on a line has usually stopped being an improvement.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

Module 06 takes the reference rule to `dict` and `set`, where it grows a condition a
list cannot meet.